In [45]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# FUNCTIONS FOR PREPROCESSING FOR TRAIN SET

In [17]:
# Split datasets

def split_sets(X, test_size=0.2, random_state=42):
    X['Date of Birth'] = pd.to_datetime(X['Date of Birth'], errors='coerce') # This is becacuse the DoB cannot be converted in X_test after splitting
    y_new = X['Survival Prediction'].copy()
    X_new = X.drop(columns=['Survival Prediction']).copy()
    X_train, X_test, y_train, y_test = train_test_split(X_new, y_new, test_size=test_size, random_state=random_state)
    y_train.replace({'No': 0, 'Yes': 1}, inplace=True)
    y_test.replace({'No': 0, 'Yes': 1}, inplace=True)
    print(f"Dimension of X_train: {X_train.shape}")
    print(f"Dimension of X_test: {X_test.shape}")
    print(f"Dimension of y_train: {y_train.shape}")
    print(f"Dimension of y_test: {y_test.shape}")
    return X_train, X_test, y_train, y_test

In [18]:
# Remove columns

# According to EDA, columns to remove: Transfusion History, Marital Status, Smoking History (due to high correlation with 'Non Smoker', plus this column has missing values)

def remove_columns(df, columns_to_delete):
  df_new = df.copy()
  for column in columns_to_delete:
    df_new = df_new.drop(columns=[column], errors='ignore')
  return df_new

In [19]:
# Data imputation for categorical variables (with the mode)

# Columns to process here: 'Healthcare Access' and 'Gender'

def df_cat_imputation(df, values_to_imput_cat):
  df_new = df.copy()
  mode_train={}
  columns_not_in_list = [x for x in df_new.select_dtypes(include = 'object').columns if x not in values_to_imput_cat.keys()]

  for column, value_to_imput in values_to_imput_cat.items():
      # Get mode from columns in values_to_imput_cat (removing rows with strange values)
        mode_1 = df_new.loc[df_new[column] != values_to_imput_cat[column], column].mode().dropna()
        if not mode_1.empty:
          mode_train[column] = mode_1[0]
          # Replace missing values
          df_new.loc[df_new[column] == values_to_imput_cat[column], column] = mode_train[column]

  # If column is not in values_to_imput_cat, save its mode as it'll be used to imput missing values in the test set.
  for column in columns_not_in_list:
        mode_2 = df_new[column].mode().dropna()
        if not mode_2.empty:
            mode_train[column] = mode_2[0]

  return df_new, mode_train

In [20]:
# Remove duplicates and rows with null values

def remove_rows(df, y_train):
    # 1. Remove duplicates
    df_no_dups = df.drop_duplicates()
    duplicates_removed = len(df) - len(df_no_dups)
    #print(f"# duplicates removed: {duplicates_removed}")

    # 2. Remove rows with missing values
    df_clean = df_no_dups.dropna()
    rows_removed = len(df_no_dups) - len(df_clean)
    #print(f"Number of rows deleted for having missing values {rows_removed}")

    # Save indices from the clean df
    surviving_indices = df_clean.index

    # Filter y_train with surviving indices
    y_clean = y_train.loc[surviving_indices]

    # Reset indices if needed
    df_clean = df_clean.reset_index(drop=False)
    y_clean = y_clean.reset_index(drop=True)

    return df_clean, y_clean

In [21]:
# Standardize values in 'Urban or Real' column

def standardize_urbal_rural (df):
  df_new = df.copy()
  df_new['Urban or Rural'] = df['Urban or Rural'].str.lower()
  df_new['Urban or Rural'] = df['Urban or Rural'].str.capitalize()

  return df_new

In [22]:
# Encode nominal variables into booleans

# Nominal columns (Yes/No): 'Diabetes History', 'Heart Disease History', 'Inflammatory Bowel Disease',
#        'Survival Prediction', 'Diabetes', 'Alcohol Consumption', 'Early Detection',
#        'Family History', 'Genetic Mutation'

def convert_into_bool(df, binary_cols):
  df_new = df.copy()
  for col in binary_cols:
        if col in df_new.columns:
            df_new[col] = df_new[col].map({'Yes': 1, 'No': 0}).astype(bool)

  return df_new

In [23]:
# Transform 'Date of Birth' column into 'Age' column

def create_age_column(df, reference_date='2025-01-01'):

      df_new = df.copy()
      if 'Date of Birth' in df_new.columns:
        # Create 'Age' Column
        try:
            df_new['Date of Birth'] = pd.to_datetime(df_new['Date of Birth'], errors='coerce')
            ref_date = pd.Timestamp(reference_date)
            df_new['Age'] = ((ref_date - df_new['Date of Birth']).dt.days / 365.25).round()
            df_new['Age'] = df_new['Age'].astype('float')

            # Delete 'Date of birth'
            df_new = df_new.drop(columns=['Date of Birth'])
            #print(f"'Date of Birth' transformed into 'Age'")
        except Exception as e:
            print(f"There was an error {e}")
      return df_new

In [24]:
# Preprocessing for numerical variables (winsorization for outliers, imputation with median)

# Numerical columns to preprocess here: 'Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K',
#        'Tumor Size (mm)', 'Age

# Column 'Healthcare Costs' has negative values. These will be imputated with the median.

# Added the argument 'scaling_true' to scale the variables using the median if needed (0 = no scaling, 1 = scaling).

def df_num_imputation(df, numeric_cols, scaling_true = 1):
  df_new = df.copy()
  stats_pre = {}

# Transform columns into float type
  for col in numeric_cols:
    if col in df_new.columns:
      df_new[col] = pd.to_numeric(df_new[col], errors='coerce')

# Remove outliers using 'Winsorization'
  for column in numeric_cols:
      Q1 = df_new[column].quantile(0.25)
      Q3 = df_new[column].quantile(0.75)
      IQR = Q3 - Q1

      lower_bound = Q1 - 1.5 * IQR
      upper_bound = Q3 + 1.5 * IQR

      # Clip outliers with upper or lower bound
      #df_new[column] = df_new[column].clip(lower=lower_bound, upper=upper_bound) # Uncomment if test fails

      # Imputing negative values with the median
      median_c = df_new[column].median()
      df_new.loc[df_new[column] < 0, column] = median_c
      stats_pre[column] = {
          'lower_bound': lower_bound,
          'upper_bound': upper_bound,
          'median': median_c
      }

  # Scaling variables using the median
  if scaling_true == 1:
      for column in numeric_cols:
          if column in stats_pre:
              median = stats_pre[column]['median']
              df_new[column] = (df_new[column] - median) / median

  # Returning the df and the stats_pre dictionary which will be used later to preprocess the test set
  return df_new, stats_pre

In [25]:
# Create dummy columns for categorical ordinal variables

# Those categoricals with less than 2 unique values will be label encoded.

# The main idea is to get rid of categorical columns by turning them into dummies

# Columns to process here
#'Cancer Stage', 'Country', 'Diet Risk', 'Gender', 'Healthcare Access',
#        'Insurance Costs', 'Insurance Status', 'Obesity BMI', 'Physical Activity',
#        'Screening History', 'Smoking History', 'Treatment Type', 'Urban or Rural'

def create_dummies(df, categorical_cols):
  df_new = df.copy()
  for col in categorical_cols:
    if col in categorical_cols:
      unique_values = df_new[col].nunique()
      if unique_values > 2:  # One-hot encoding will be performed on variables with more than 2 unique values
        dummies = pd.get_dummies(df_new[col], prefix=col, drop_first=False)
        df_new = pd.concat([df_new, dummies], axis=1)
        df_new = df_new.drop(columns=[col])  # Drop original column after getting dummies

  # Identify remaining categorical columns

  non_numeric_cols = []
  for col in df_new.columns:
      if df_new[col].dtype == 'object':
          #print(f"Non numeric column found: {col}")
          non_numeric_cols.append(col)

  # Transform previous columns

  for col in non_numeric_cols:
    # Customized code for 'Gender' column
    if col in non_numeric_cols:
      if col == 'Gender' and 'Gender' in non_numeric_cols:
        df_new['Gender'] = df_new['Gender'].map({'M': 0, 'F': 1}).astype(bool)
      else:
        dummies = pd.get_dummies(df_new[col], prefix=col, drop_first=True).astype(bool)
        df_new = pd.concat([df_new, dummies], axis=1)
        df_new = df_new.drop(columns=[col])

  # Check for non-numeric remaining columns

  for col in df_new.columns:
    if df_new[col].dtype == 'object':
      print(f"Error: Column {col} remains as non-numeric")

  return df_new

In [26]:
def balance_features(X_train, y_train, columns_to_balance, random_state=42):

    # Reset indices to ensure alignment
    X_result = X_train.copy().reset_index(drop=True)
    y_result = y_train.copy().reset_index(drop=True)

    for col in columns_to_balance:
        print(f"\nBalancing column: {col}")
        print(f"Original distribution: {X_result[col].value_counts().to_dict()}")

        # Apply SMOTE
        smote = SMOTE(random_state=random_state)
        features = X_result.drop(columns=col)
        target = X_result[col]

        # Resampling
        features_balanced, target_balanced = smote.fit_resample(features, target)

        # Calculate how many synthetic samples were generated
        n_original = len(X_result)
        n_balanced = len(features_balanced)
        n_synthetic = n_balanced - n_original

        # Rebuild X_balanced
        X_balanced = pd.DataFrame(features_balanced, columns=features.columns)
        X_balanced[col] = target_balanced
        X_balanced = X_balanced.reset_index(drop=True)

        # Generate new values for y_balanced
        if n_synthetic > 0:
            # Generate synthetic values following the original distribution
            if isinstance(y_result, pd.Series):
                class_distribution = y_result.value_counts(normalize=True)
                classes = class_distribution.index.tolist()
                probs = class_distribution.values

                y_synthetic = pd.Series(
                    np.random.choice(classes, size=n_synthetic, p=probs),
                    name=y_result.name
                )

                y_balanced = pd.concat([y_result, y_synthetic]).reset_index(drop=True)

            else:  # DataFrame
                y_synthetic = pd.DataFrame(index=range(n_synthetic))

                for y_col in y_result.columns:
                    class_distribution = y_result[y_col].value_counts(normalize=True)
                    classes = class_distribution.index.tolist()
                    probs = class_distribution.values

                    y_synthetic[y_col] = np.random.choice(classes, size=n_synthetic, p=probs)

                y_balanced = pd.concat([y_result, y_synthetic]).reset_index(drop=True)

        else:
            y_balanced = y_result.copy()

        # Update for next iteration
        X_result = X_balanced
        y_result = y_balanced

        print(f"Distribution after SMOTE: {X_result[col].value_counts().to_dict()}")
        print(f"Dimensions: X={X_result.shape}, y={y_result.shape}")

    return X_result, y_result

In [27]:
# Macro function for gathering the previous ones

def preprocess_train_df(X, y, columns_to_delete, values_to_imput_cat, binary_cols, numeric_cols , categorical_cols):
  # Create a new feature 'Cardiometabolic_Risk' based on 'Diabetes' and 'Heart Disease History'
  X_new = X.copy()
  X_new['Cardiometabolic_Risk'] = ((X_new['Diabetes'] == 'Yes') | (X_new ['Heart Disease History'] == 'Yes')).astype(int)

  X_clean = remove_columns(X_new, columns_to_delete) # Goes first because there are columns with empty values or that are irrelevant.
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean, mode_train = df_cat_imputation(X_clean, values_to_imput_cat) # Goes second because rows with values to be imputed are removed afterwards
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean, y_train_processed = remove_rows(X_clean, y)
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean = standardize_urbal_rural(X_clean)
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean = convert_into_bool(X_clean, binary_cols)
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean = create_age_column(X_clean)
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean, stats_pre = df_num_imputation(X_clean, numeric_cols)
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_train_processed = create_dummies(X_clean, categorical_cols)
  print(f"Dimension of X_clean after: {X_train_processed.shape}")
  X_train_processed.reset_index(drop=True, inplace=True)
  print(f"Dimension of X_clean after: {X_train_processed.shape}")
  X_train_processed.drop(columns='ID', inplace=True)
  #X_train_balanced, y_train_balanced = balance_features(X_train_processed, y_train_processed, columns_to_balance)


  #return X_train_balanced, y_train_balanced, mode_train, stats_pre
  return X_train_processed, y_train_processed, mode_train, stats_pre

# FUNCTIONS FOR PREPROCESSING FOR TEST SET

In [28]:
def testdf_categorical_imputation(df, values_to_imput_cat, mode_train):

  df_new = df.copy()
  categorical_columns = df_new.select_dtypes(include=['object', 'category']).columns

  # Replacing strange values of columns in values_to_imput_cat with the mode stored in mode_train dictionary
  for column, value_to_imput in values_to_imput_cat.items():
    if column in mode_train: # Verifying if the column exists in mode_train as a key
      df_new.loc[df_new[column] == values_to_imput_cat[column], column] = mode_train[column] # Replacing strange value with the mode of the column

  # Replacing missing values with the mode stored in mode_train dictionary
  for column in categorical_columns:
    if column in mode_train: # Verifying if the column exists in mode_train as a key
      df_new[column] = df_new[column].fillna(mode_train[column]) # Filling NaNs with the mode

  return df_new

In [29]:
#Added the argument 'scaling_true' to scale the variables using the median if needed (0 = no scaling, 1 = scaling).

def testdf_numeric_imputation(df, stats_pre, numeric_cols, scaling_true = 1):
  df_new = df.copy()

# Transform columns into float type
  for column in numeric_cols:
    if column in df_new.columns:
      df_new[column] = pd.to_numeric(df_new[column], errors='coerce')

# Remove outliers using 'Winsorization'
  for column in numeric_cols:

      # Retrieving values from stats_pre dictionary. In case they do not exist, variables will get 'None' value
      lower_bound = stats_pre[column].get('lower_bound', None)
      upper_bound = stats_pre[column].get('upper_bound', None)
      median = stats_pre[column].get('median', None)

      if lower_bound is not None and upper_bound is not None:
        # Clipping datapoint outside of the range
        df_new[column] = df_new[column].clip(lower=lower_bound, upper=upper_bound)

      if median is not None:
        # Imputing missing values with the median
        df_new[column] = df_new[column].fillna(median)

  # Scaling variables using the median
  if scaling_true == 1:
    for column in numeric_cols:
        if column in stats_pre:
            median = stats_pre[column]['median']
            df_new[column] = (df_new[column] - median) / median

  return df_new

In [30]:
def preprocess_test_df(X, X_train_columns, values_to_imput_cat, columns_to_delete, mode_train, stats_pre, binary_cols, numeric_cols, categorical_cols):
  # Create a new feature 'Cardiometabolic_Risk' based on 'Diabetes' and 'Heart Disease History'
  X['Cardiometabolic_Risk'] = ((X['Diabetes'] == 'Yes') | (X['Heart Disease History'] == 'Yes')).astype(int)

  X_clean = remove_columns(X, columns_to_delete)
  X_clean = standardize_urbal_rural(X_clean)
  X_clean = testdf_categorical_imputation(X_clean, values_to_imput_cat, mode_train)
  X_clean = convert_into_bool(X_clean, binary_cols)
  X_clean = create_age_column(X_clean) # Null values in Age because missing DoB should be imputed with the median next.
  X_clean = testdf_numeric_imputation(X_clean, stats_pre, numeric_cols)
  X_test_processed = create_dummies(X_clean, categorical_cols)
  X_test_processed.reset_index(drop=True, inplace=True)
  X_test_processed = X_test_processed[X_train_columns]

  return X_test_processed

In [31]:
__all__ = [
    # Variables
    "columns_to_delete",
    "values_to_imput_cat",
    "binary_cols",
    "numeric_cols",
    "categorical_cols",
    "columns_to_balance",

    # Functions
    "split_sets",
    "remove_columns",
    "df_cat_imputation",
    "remove_rows",
    "standardize_urbal_rural",
    "convert_into_bool",
    "create_age_column",
    "df_num_imputation",
    "create_dummies",
    "balance_features",
    "preprocess_train_df",
    "preprocess_test_df",
    "testdf_categorical_imputation",
    "testdf_numeric_imputation"
]

# PARAMETERS

In [32]:
columns_to_delete = ['Transfusion History', 'Marital Status', 'Smoking History', 'Diabetes History']

values_to_imput_cat = {
        'Healthcare Access': '?',
        'Gender': 'P'
    }

binary_cols = [
        'Heart Disease History', 'Inflammatory Bowel Disease',
        'Diabetes', 'Alcohol Consumption', 'Early Detection',
        'Family History', 'Genetic Mutation'
    ]

numeric_cols = [
        'Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K',
        'Tumor Size (mm)', 'Age'
    ]

categorical_cols = [
        'Cancer Stage', 'Country', 'Diet Risk', 'Gender', 'Healthcare Access',
        'Insurance Costs', 'Insurance Status', 'Obesity BMI', 'Physical Activity',
        'Screening History', 'Non Smoker', 'Treatment Type', 'Urban or Rural'
]

#columns_to_balance = ['Diabetes History', 'Heart Disease History', 'Inflammatory Bowel Disease']

mode_train = {}

stats_pre = {}

# PREPROCESSING TRAIN SET

In [33]:
cancer_df = pd.read_csv('https://raw.githubusercontent.com/gascalero/DM_II_project/4a73ca4928f2b95f960cd9b9f44c4700244ed553/data/raw/patient_train_data.csv',
                        encoding='UTF-8',
                        index_col=0,
                        sep=',',
                        on_bad_lines='skip',
                        quoting=3)
cancer_df.head(1)

,Alcohol Consumption,Cancer Stage,Country,Date of Birth,Diabetes,Diabetes History,Diet Risk,Early Detection,Family History,Gender,...,Non Smoker,Obesity BMI,Physical Activity,Screening History,Smoking History,Transfusion History,Treatment Type,Tumor Size (mm),Urban or Rural,Survival Prediction
ID,,,,,,,,,,,,,,,,,,,,,
1,No,Localized,UK,29-01-1966,No,No,Moderate,No,No,M,...,Yes,Overweight,Low,Regular,No,-,Chemotherapy,33.0,Urban,Yes


In [34]:
X_train, X_test, y_train, y_test = split_sets(cancer_df)

Dimension of X_train: (60028, 30)
Dimension of X_test: (15007, 30)
Dimension of y_train: (60028,)
Dimension of y_test: (15007,)


In [35]:
X_train_processed, y_train_processed, mode_train, stats_pre = preprocess_train_df(X_train, y_train, columns_to_delete, values_to_imput_cat, binary_cols, numeric_cols, categorical_cols)

Dimension of X_clean after: (60028, 27)
Dimension of X_clean after: (60028, 27)
Dimension of X_clean after: (59180, 28)
Dimension of X_clean after: (59180, 28)
Dimension of X_clean after: (59180, 28)
Dimension of X_clean after: (59180, 28)
Dimension of X_clean after: (59180, 28)
Dimension of X_clean after: (59180, 60)
Dimension of X_clean after: (59180, 60)


In [36]:
X_val_processed = preprocess_test_df(X_test, X_train_processed.columns, values_to_imput_cat, columns_to_delete, mode_train, stats_pre, binary_cols, numeric_cols, categorical_cols)

y_val_processed = y_test.copy()

# MODEL EVALUATION AND SELECTION

In [43]:
from collections import Counter
import copy
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import cross_val_score, KFold, cross_validate
#import optuna
#from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostClassifier
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import optuna
from optuna.samplers import TPESampler
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore")


%matplotlib inline

## DUMMY CLASSIFIER

In [47]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

zero_r = DummyClassifier(strategy="most_frequent")
zero_r.fit(X_train_processed, y_train_processed)

y_pred_zeror = zero_r.predict(X_val_processed)
y_pred_zeror_train = zero_r.predict(X_train_processed)

# Evaluar el modelo
print("ZeroR Classification Report (test set):")
print(classification_report(y_val_processed, y_pred_zeror))

print("ZeroR Classification Report (train set):")
print(classification_report(y_train_processed, y_pred_zeror_train))

ZeroR Classification Report (test set):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      5976
           1       0.60      1.00      0.75      9031

    accuracy                           0.60     15007
   macro avg       0.30      0.50      0.38     15007
weighted avg       0.36      0.60      0.45     15007

ZeroR Classification Report (train set):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     23675
           1       0.60      1.00      0.75     35505

    accuracy                           0.60     59180
   macro avg       0.30      0.50      0.37     59180
weighted avg       0.36      0.60      0.45     59180



## LOGISTIC REGRESSION

In [48]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(C=0.2323350351539011, class_weight='balanced', max_iter=331182, penalty='l1', solver='liblinear')
#logistic_model = LogisticRegression(class_weight='balanced', max_iter=100000, random_state=42)

logistic_model.fit(X_train_processed, y_train_processed)

y_pred_logistic = logistic_model.predict(X_val_processed)
y_pred_logistic_train = logistic_model.predict(X_train_processed)

print("Classification (test) Report:")
print(classification_report(y_val_processed, y_pred_logistic))

print("Classification (training) Report:")
print(classification_report(y_train_processed, y_pred_logistic_train))

Classification (test) Report:
              precision    recall  f1-score   support

           0       0.39      0.47      0.43      5976
           1       0.60      0.51      0.55      9031

    accuracy                           0.50     15007
   macro avg       0.49      0.49      0.49     15007
weighted avg       0.51      0.50      0.50     15007

Classification (training) Report:
              precision    recall  f1-score   support

           0       0.41      0.50      0.45     23675
           1       0.61      0.53      0.57     35505

    accuracy                           0.52     59180
   macro avg       0.51      0.51      0.51     59180
weighted avg       0.53      0.52      0.52     59180



## KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(weights='distance', metric = 'manhattan', n_neighbors=9)

knn_model.fit(X_train_processed, y_train_processed)

y_pred_knn = knn_model.predict(X_test_processed)
y_pred_knn_train = knn_model.predict(X_train_processed)

print("Classification (test) Report:")
print(classification_report(y_test_processed, y_pred_knn))

print("XGBoost Classification (training) Report:")
print(classification_report(y_train_processed, y_pred_knn_train))

Classification (test) Report:
              precision    recall  f1-score   support

           0       0.40      0.27      0.32      5976
           1       0.60      0.73      0.66      9031

    accuracy                           0.55     15007
   macro avg       0.50      0.50      0.49     15007
weighted avg       0.52      0.55      0.52     15007

XGBoost Classification (training) Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     23675
           1       1.00      1.00      1.00     35505

    accuracy                           1.00     59180
   macro avg       1.00      1.00      1.00     59180
weighted avg       1.00      1.00      1.00     59180



## RANDOM FOREST CLASSIFIER

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    bootstrap=True,
    class_weight='balanced',
    random_state=42
)

random_forest.fit(X_train_processed, y_train_processed)

y_pred_rf = random_forest.predict(X_test_processed)
y_pred_rf_train = random_forest.predict(X_train_processed)

print("Classification (test) Report:")
print(classification_report(y_test_processed, y_pred_rf))

print("Classification (training) Report:")
print(classification_report(y_train_processed, y_pred_rf_train))

Classification (test) Report:
              precision    recall  f1-score   support

           0       0.40      0.39      0.40      5976
           1       0.61      0.62      0.61      9031

    accuracy                           0.53     15007
   macro avg       0.50      0.50      0.50     15007
weighted avg       0.53      0.53      0.53     15007

Classification (training) Report:
              precision    recall  f1-score   support

           0       0.72      0.74      0.73     23675
           1       0.82      0.81      0.82     35505

    accuracy                           0.78     59180
   macro avg       0.77      0.78      0.77     59180
weighted avg       0.78      0.78      0.78     59180



## XGBOOST

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42)
xgb_model.set_params(
    n_estimators=100,
    max_depth=10,
    min_child_weight=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    scale_pos_weight=1,  # Adjust this based on class imbalance
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train_processed, y_train_processed)

y_pred_xg = xgb_model.predict(X_test_processed)
y_pred_xg_train = xgb_model.predict(X_train_processed)

print("Classification (test) Report:")
print(classification_report(y_test_processed, y_pred_rf))

print("Classification (training) Report:")
print(classification_report(y_train_processed, y_pred_rf_train))


Classification (test) Report:
              precision    recall  f1-score   support

           0       0.40      0.39      0.40      5976
           1       0.61      0.62      0.61      9031

    accuracy                           0.53     15007
   macro avg       0.50      0.50      0.50     15007
weighted avg       0.53      0.53      0.53     15007

Classification (training) Report:
              precision    recall  f1-score   support

           0       0.72      0.74      0.73     23675
           1       0.82      0.81      0.82     35505

    accuracy                           0.78     59180
   macro avg       0.77      0.78      0.77     59180
weighted avg       0.78      0.78      0.78     59180



# GRIDSEARCH

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score

param_grid_simplified = {
    'C': [0.01, 0.1, 1, 10, 20, 50, 100],
    'penalty': ['l2'],
    'solver': ['liblinear', 'lbfgs', 'saga'],
    'class_weight': ['balanced'],
    'max_iter': [1000000]
}

# Crear el modelo base
log_reg = LogisticRegression(random_state=42)

# Definir qué métrica optimizar (F1-score es bueno para datos desbalanceados)
scorer = make_scorer(f1_score)

# Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid_simplified,
    scoring=scorer,
    cv=5,  # Validación cruzada de 5 folds
    n_jobs=-1,  # Usar todos los núcleos disponibles
    verbose=2,   # Mostrar progreso
    return_train_score=True
)

# Ajustar GridSearch a los datos
grid_search.fit(X_train_processed, y_train_processed)

# Ver los mejores parámetros
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print(f"Mejor puntuación F1: {grid_search.best_score_:.4f}")

# Crear modelo con los mejores parámetros
best_log_reg = LogisticRegression(**grid_search.best_params_, random_state=42)

# Entrenar con todos los datos
best_log_reg.fit(X_train_processed, y_train_processed)

# Evaluar en conjunto de prueba
y_pred_best = best_log_reg.predict(X_val_processed)
print("\nInforme de clasificación con los mejores parámetros:")
print(classification_report(y_val_processed, y_pred_best))

# Probabilidades para calcular AUC/ROC si es necesario
#y_prob_best = best_log_reg.predict_proba(X_test_processed)[:, 1]
#roc_auc = roc_auc_score(y_test_processed, y_prob_best)
#print(f"ROC AUC Score: {roc_auc:.4f}")

Fitting 5 folds for each of 21 candidates, totalling 105 fits
Mejores parámetros encontrados:
{'C': 20, 'class_weight': 'balanced', 'max_iter': 1000000, 'penalty': 'l2', 'solver': 'liblinear'}
Mejor puntuación F1: 0.5557

Informe de clasificación con los mejores parámetros:
              precision    recall  f1-score   support

           0       0.39      0.48      0.43      5976
           1       0.60      0.51      0.55      9031

    accuracy                           0.50     15007
   macro avg       0.49      0.49      0.49     15007
weighted avg       0.51      0.50      0.50     15007



# OPTUNA

In [3]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 13.8 MB/s eta 0:00:00


In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from imblearn.over_sampling import SMOTE
from collections import Counter
import copy
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import cross_val_score, KFold, cross_validate
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostClassifier
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore")

In [38]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

## LOGISTIC REGRESSION - OPTUNA

In [59]:
def logistic_regression_objective(trial):
    # Parámetros para optimizar en Logistic Regression
    C = trial.suggest_float('C', 0.1, 100.0, log=True)
    solver = trial.suggest_categorical('solver', ['liblinear', 'saga'])
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
    max_iter = trial.suggest_int('max_iter', 100000, 1000000, log=True)

    # Crear el modelo Logistic Regression
    lr_model = LogisticRegression(
        C=C,
        solver=solver,
        penalty=penalty,
        class_weight=class_weight,
        max_iter=max_iter,
        random_state=42,
        n_jobs=4
    )

    scoring = {
        'accuracy': 'accuracy',
        'f1_weighted': 'f1_weighted'
    }

    cv_results = cross_validate(
        lr_model,
        X_train_smote,
        y_train_smote,
        cv=kf,
        scoring=scoring,
        return_train_score=True
    )

    train_acc = cv_results['train_accuracy'].mean()
    test_acc = cv_results['test_accuracy'].mean()
    test_f1 = cv_results['test_f1_weighted'].mean()

    overfit = train_acc - test_acc

    if overfit > 0.05:
        penalty = (overfit - 0.05) * 2
        objective_score = test_f1 - penalty
    else:
        objective_score = test_f1

    trial.set_user_attr('train_accuracy', train_acc)
    trial.set_user_attr('test_accuracy', test_acc)
    trial.set_user_attr('test_f1', test_f1)
    trial.set_user_attr('overfit', overfit)

    return objective_score

def optimize_lr_model(n_trials=20):
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
    )

    def print_trial_callback(study, trial):
        print(f"\nTrial {trial.number}:")
        print(f"    Params: {trial.params}")
        print(f"    Training accuracy: {trial.user_attrs['train_accuracy']:.4f}")
        print(f"    Validation accuracy: {trial.user_attrs['test_accuracy']:.4f}")
        print(f"    Validation F1: {trial.user_attrs['test_f1']:.4f}")
        print(f"    Overfitting: {trial.user_attrs['overfit']:.4f}")
        print(f"    Objective value: {trial.value:.4f}")

    study.optimize(logistic_regression_objective, n_trials=n_trials, callbacks=[print_trial_callback])

    best_params = study.best_params
    best_trial = study.best_trial

    print("Mejores hiperparámetros encontrados para Logistic Regression:")
    for param, value in best_params.items():
        print(f"    {param}: {value}")

    print("\nMétricas de validación cruzada del mejor modelo:")
    print(f"    Accuracy en entrenamiento: {best_trial.user_attrs['train_accuracy']:.4f}")
    print(f"    Accuracy en validación: {best_trial.user_attrs['test_accuracy']:.4f}")
    print(f"    F1 score en validación: {best_trial.user_attrs['test_f1']:.4f}")
    print(f"    Sobreajuste: {best_trial.user_attrs['overfit']:.4f}")

    return best_params, study

In [58]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train_processed)

In [ ]:
# Para ejecutar la optimización:
print("\nOptimizando Logistic Regression...")
lr_best_params, lr_study = optimize_lr_model(n_trials=20)  # Reducido a 10 para mayor velocidad

# Crear y entrenar el mejor modelo
best_lr = LogisticRegression(
    C=lr_best_params['C'],
    solver=lr_best_params['solver'],
    penalty=lr_best_params['penalty'],
    class_weight=lr_best_params['class_weight'],
    max_iter=lr_best_params['max_iter'],
    random_state=42,
    n_jobs=-1  # Usar todos los cores
)

best_lr.fit(X_train_smote, y_train_smote)
lr_y_pred_train = best_lr.predict(X_train_smote)
lr_y_pred_test = best_lr.predict(X_val_processed)

lr_train_acc = accuracy_score(y_train_smote, lr_y_pred_train)
lr_test_acc = accuracy_score(y_val_processed, lr_y_pred_test)
lr_test_f1 = f1_score(y_val_processed, lr_y_pred_test, average='weighted')
lr_overfit = lr_train_acc - lr_test_acc

print("\nMétricas del modelo Logistic Regression final:")
print(f"    Accuracy en entrenamiento: {lr_train_acc:.4f}")
print(f"    Accuracy en prueba: {lr_test_acc:.4f}")
print(f"    F1 score en prueba: {lr_test_f1:.4f}")
print(f"    Sobreajuste: {lr_overfit:.4f}")

[I 2025-05-02 15:31:08,202] A new study created in memory with name: no-name-6660b4fe-6646-4b2f-9eda-0d661e391ee9



Optimizando Logistic Regression...


## XGBOOST - OPTUNA

In [55]:
def xgboost_objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 150, 800)
    max_depth = trial.suggest_int('max_depth', 1, 3)
    learning_rate = trial.suggest_float('learning_rate', 0.0005, 0.01, log=True)
    subsample = trial.suggest_float('subsample', 0.5, 0.8)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 0.8)
    min_child_weight = trial.suggest_int('min_child_weight', 10, 50)
    gamma = trial.suggest_float('gamma', 0.5, 5.0)
    alpha = trial.suggest_float('alpha', 5.0, 20.0)
    lambda_param = trial.suggest_float('lambda', 5.0, 20.0)


    xgb_model = xgb.XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma,
        reg_alpha=alpha,
        reg_lambda=lambda_param,
        random_state=42,
        n_jobs=4,
        use_label_encoder=False,
        eval_metric='logloss'
    )

    scoring = {
        'accuracy': 'accuracy',
        'f1_weighted': 'f1_weighted'
    }

    cv_results = cross_validate(
        xgb_model,
        X_train_processed,
        y_train_processed,
        cv=kf,
        scoring=scoring,
        return_train_score=True
    )

    train_acc = cv_results['train_accuracy'].mean()
    test_acc = cv_results['test_accuracy'].mean()
    test_f1 = cv_results['test_f1_weighted'].mean()

    overfit = train_acc - test_acc

    if overfit > 0.05:
        penalty = (overfit - 0.05) * 2
        objective_score = test_f1 - penalty
    else:
        objective_score = test_f1

    trial.set_user_attr('train_accuracy', train_acc)
    trial.set_user_attr('test_accuracy', test_acc)
    trial.set_user_attr('test_f1', test_f1)
    trial.set_user_attr('overfit', overfit)

    return objective_score

def optimize_xgb_model(n_trials=20):
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
    )

    def print_trial_callback(study, trial):
        print(f"\nTrial {trial.number}:")
        print(f"    Params: {trial.params}")
        print(f"    Training accuracy: {trial.user_attrs['train_accuracy']:.4f}")
        print(f"    Validation accuracy: {trial.user_attrs['test_accuracy']:.4f}")
        print(f"    Validation F1: {trial.user_attrs['test_f1']:.4f}")
        print(f"    Overfitting: {trial.user_attrs['overfit']:.4f}")
        print(f"    Objective value: {trial.value:.4f}")

    study.optimize(xgboost_objective, n_trials=n_trials, callbacks=[print_trial_callback])

    best_params = study.best_params
    best_trial = study.best_trial

    print("Mejores hiperparámetros encontrados para XGBoost:")
    for param, value in best_params.items():
        print(f"    {param}: {value}")

    print("\nMétricas de validación cruzada del mejor modelo:")
    print(f"    Accuracy en entrenamiento: {best_trial.user_attrs['train_accuracy']:.4f}")
    print(f"    Accuracy en validación: {best_trial.user_attrs['test_accuracy']:.4f}")
    print(f"    F1 score en validación: {best_trial.user_attrs['test_f1']:.4f}")
    print(f"    Sobreajuste: {best_trial.user_attrs['overfit']:.4f}")

    return best_params, study

In [56]:
print("\nOptimizando XGBoost...")
xgboost_best_params, xgboost_study = optimize_xgb_model(n_trials=20)

best_xgb = xgb.XGBClassifier(
    n_estimators=xgboost_best_params['n_estimators'],
    max_depth=xgboost_best_params['max_depth'],
    learning_rate=xgboost_best_params['learning_rate'],
    subsample=xgboost_best_params['subsample'],
    colsample_bytree=xgboost_best_params['colsample_bytree'],
    min_child_weight=xgboost_best_params['min_child_weight'],
    gamma=xgboost_best_params['gamma'],
    reg_alpha=xgboost_best_params['alpha'],
    reg_lambda=xgboost_best_params['lambda'],
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss'
)

best_xgb.fit(X_train_processed, y_train_processed)
xgb_y_pred_train = best_xgb.predict(X_train_processed)
xgb_y_pred_test = best_xgb.predict(X_val_processed)

xgb_train_acc = accuracy_score(y_train_processed, xgb_y_pred_train)
xgb_test_acc = accuracy_score(y_val_processed, xgb_y_pred_test)
xgb_test_f1 = f1_score(y_test, xgb_y_pred_test, average='weighted')
xgb_overfit = xgb_train_acc - xgb_test_acc

print("\nMétricas del modelo XGBoost final:")
print(f"    Accuracy en entrenamiento: {xgb_train_acc:.4f}")
print(f"    Accuracy en prueba: {xgb_test_acc:.4f}")
print(f"    F1 score en prueba: {xgb_test_f1:.4f}")
print(f"    Sobreajuste: {xgb_overfit:.4f}")

[I 2025-05-02 14:40:31,019] A new study created in memory with name: no-name-8c1cfe8c-100d-40d3-8b44-e9f0ec8df55c



Optimizando XGBoost...


[I 2025-05-02 14:40:50,772] Trial 0 finished with value: 0.4499421250970091 and parameters: {'n_estimators': 393, 'max_depth': 3, 'learning_rate': 0.0044803926826840635, 'subsample': 0.679597545259111, 'colsample_bytree': 0.5468055921327309, 'min_child_weight': 16, 'gamma': 0.7613762547568976, 'alpha': 17.99264218662403, 'lambda': 14.016725176148132}. Best is trial 0 with value: 0.4499421250970091.



Trial 0:
    Params: {'n_estimators': 393, 'max_depth': 3, 'learning_rate': 0.0044803926826840635, 'subsample': 0.679597545259111, 'colsample_bytree': 0.5468055921327309, 'min_child_weight': 16, 'gamma': 0.7613762547568976, 'alpha': 17.99264218662403, 'lambda': 14.016725176148132}
    Training accuracy: 0.5999
    Validation accuracy: 0.5999
    Validation F1: 0.4499
    Overfitting: 0.0000
    Objective value: 0.4499


[I 2025-05-02 14:41:07,281] Trial 1 finished with value: 0.4499421250970091 and parameters: {'n_estimators': 610, 'max_depth': 1, 'learning_rate': 0.009138013915892867, 'subsample': 0.7497327922401266, 'colsample_bytree': 0.5637017332034828, 'min_child_weight': 17, 'gamma': 1.325320294340452, 'alpha': 9.563633644393066, 'lambda': 12.871346474483568}. Best is trial 0 with value: 0.4499421250970091.


...


Trial 10:
    Params: {'n_estimators': 765, 'max_depth': 2, 'learning_rate': 0.0012568672294227042, 'subsample': 0.7796871025735361, 'colsample_bytree': 0.6564020035588104, 'min_child_weight': 38, 'gamma': 0.5681490669624059, 'alpha': 14.350647779441982, 'lambda': 13.558217973225247}
    Training accuracy: 0.5999
    Validation accuracy: 0.5999
    Validation F1: 0.4499
    Overfitting: 0.0000
    Objective value: 0.4499


[I 2025-05-02 14:44:12,985] Trial 11 finished with value: 0.4499421250970091 and parameters: {'n_estimators': 689, 'max_depth': 2, 'learning_rate': 0.00905934967030795, 'subsample': 0.7611105739055738, 'colsample_bytree': 0.6478593110951146, 'min_child_weight': 20, 'gamma': 1.5039270787215264, 'alpha': 10.999010191513522, 'lambda': 13.403746254548363}. Best is trial 0 with value: 0.4499421250970091.



Trial 11:
    Params: {'n_estimators': 689, 'max_depth': 2, 'learning_rate': 0.00905934967030795, 'subsample': 0.7611105739055738, 'colsample_bytree': 0.6478593110951146, 'min_child_weight': 20, 'gamma': 1.5039270787215264, 'alpha': 10.999010191513522, 'lambda': 13.403746254548363}
    Training accuracy: 0.5999
    Validation accuracy: 0.5999
    Validation F1: 0.4499
    Overfitting: 0.0000
    Objective value: 0.4499


[W 2025-05-02 14:44:24,591] Trial 12 failed with parameters: {'n_estimators': 276, 'max_depth': 3, 'learning_rate': 0.008780983441254853, 'subsample': 0.7163128012548506, 'colsample_bytree': 0.6406115656663527, 'min_child_weight': 21, 'gamma': 1.6134087992117083, 'alpha': 12.64609698351559, 'lambda': 11.304975495124566} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "<ipython-input-55-ff90208a032f>", line 34, in xgboost_objective
    cv_results = cross_validate(
                 ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/utils/_param_validation.py", line 216, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 411, in cross_vali

KeyboardInterrupt: 

# OTRA COSA (NO TOCAR HASTA CUANDO HAYAS SELECCIONADO MODELO Y PARÁMETROS)

In [49]:
test_cancer_df = pd.read_csv('https://raw.githubusercontent.com/gascalero/DM_II_project/refs/heads/master/data/raw/patient_test_data.csv',
                         encoding='UTF-8',
                         index_col=0,
                         sep=',',
                         on_bad_lines='skip',
                         quoting=3)

test_cancer_df.head(1)

,Alcohol Consumption,Cancer Stage,Country,Date of Birth,Diabetes,Diabetes History,Diet Risk,Early Detection,Family History,Gender,...,Mortality Rate per 100K,Non Smoker,Obesity BMI,Physical Activity,Screening History,Smoking History,Transfusion History,Treatment Type,Tumor Size (mm),Urban or Rural
ID,,,,,,,,,,,,,,,,,,,,,
75036,Yes,Localized,UK,17-11-1947,No,No,Low,Yes,No,M,...,5.0,Yes,Overweight,Low,Regular,No,-,Combination,69.0,Urban


In [50]:
X_test_processed = preprocess_test_df(test_cancer_df, X_train_processed.columns, values_to_imput_cat, columns_to_delete, mode_train, stats_pre, binary_cols, numeric_cols, categorical_cols)

#X_test_processed, y_test_processed = preprocess_test_df(test_cancer_df, y_test, X_train_processed.columns, values_to_imput_cat, columns_to_delete, mode_train, stats_pre, binary_cols, numeric_cols, categorical_cols)

In [51]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(C=0.2323350351539011, class_weight='balanced', max_iter=331182, penalty='l1', solver='liblinear')
#logistic_model = LogisticRegression(C=10, class_weight='balanced', max_iter=100000, penalty='l2', solver='liblinear')
#logistic_model = LogisticRegression(class_weight='balanced', max_iter=100000)

logistic_model.fit(X_train_processed, y_train_processed)
y_pred_logistic = logistic_model.predict(X_test_processed)

In [52]:
df_kaggle = pd.DataFrame(y_pred_logistic, index=test_cancer_df.index)

df_kaggle.replace({0: 'No', 1: 'Yes'}, inplace = True)

df_kaggle.columns = ['Survival Prediction']

df_kaggle.value_counts()

,count
Survival Prediction,
Yes,39197
No,35803


In [53]:
df_kaggle.to_csv('DT_Group05_Version06.csv')

In [57]:
from google.colab import files
files.download('DT_Group05_Version06.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#X_train_processed.describe().T